In [22]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister

from encoder import encode
from encoder_advanced import prepare_state
from syndrome import extract_syndrome
from recovery import recover
from decode import decode
from measurement import measure
from measurement_advanced import measure_state

#creg = ClassicalRegister(5)
#qc = QuantumCircuit(5, creg)


q = QuantumRegister(5, "q")

syn = ClassicalRegister(2, "syn")
out = ClassicalRegister(1, "out")

qc = QuantumCircuit(q, syn, out)

state = "-"
#encode(qc)
prepare_state(qc, state)

# idle period
qc.id(0)
qc.id(1)
qc.id(2)
#qc.z(2)


extract_syndrome(qc, state, syn)

recover(qc, state, syn)

decode(qc, state)

#measure(qc)
measure_state(qc, state, out)

In [23]:
from qiskit_aer.noise import NoiseModel, pauli_error

noise_model = NoiseModel()

p = 0.2

if state in ["0", "1"]:
    error = pauli_error([
       ("X", p),
        ("I", 1. - p)])
elif state in ["+", "-"]:
    error = pauli_error([
       ("Z", p),
        ("I", 1. - p)])

# Apply to identity gates
noise_model.add_all_qubit_quantum_error(error, ["id"])

In [24]:
from qiskit import transpile
from qiskit_aer import AerSimulator


sim = AerSimulator(noise_model=noise_model)

compiled = transpile(qc, sim, optimization_level=0)

result = sim.run(compiled, shots=1024).result()

In [25]:
print(qc.draw())

       ┌───┐          ┌───┐┌───┐┌───┐                                       »
  q_0: ┤ X ├──■────■──┤ H ├┤ I ├┤ X ├───────────────────────────────────────»
       └───┘┌─┴─┐  │  ├───┤├───┤└─┬─┘┌───┐     ┌───┐                        »
  q_1: ─────┤ X ├──┼──┤ H ├┤ I ├──┼──┤ X ├─────┤ X ├────────────────────────»
            └───┘┌─┴─┐├───┤├───┤  │  └─┬─┘     └─┬─┘┌───┐           ┌────── »
  q_2: ──────────┤ X ├┤ H ├┤ I ├──┼────┼─────────┼──┤ X ├───────────┤ If-0  »
       ┌───┐     └───┘└───┘└───┘  │    │  ┌───┐  │  └─┬─┘┌─┐        └──╥─── »
  q_3: ┤ H ├──────────────────────■────■──┤ H ├──┼────┼──┤M├───────────╫────»
       ├───┤                              └───┘  │    │  └╥┘┌───┐┌─┐   ║    »
  q_4: ┤ H ├─────────────────────────────────────■────■───╫─┤ H ├┤M├───╫────»
       └───┘                                              ║ └───┘└╥┘┌──╨──┐ »
syn: 2/═══════════════════════════════════════════════════╩═══════╩═╡ 0x2 ╞═»
                                                          0     

In [26]:
counts = result.get_counts()
print(counts)

{'0 00': 9, '0 10': 37, '1 01': 144, '0 11': 36, '1 11': 140, '1 00': 511, '0 01': 28, '1 10': 119}


In [27]:
logical_errors = 0
shots = 0

for bitstring, count in counts.items():

    logical = bitstring[0]      # adjust indexing as needed

    shots += count
    if state in ["1", "-"]:
      if logical != "1":
         logical_errors += count
    else:
      if logical != "0":
         logical_errors += count  

logical_error_rate = logical_errors / shots

In [28]:
print(logical_error_rate)

0.107421875


In [8]:
print(compiled)

global phase: π/2
                                         ┌───────┐    ┌────┐   ┌────────┐┌────┐»
  q_0: ─────■─────────────────────■──────┤ Rz(π) ├────┤ √X ├───┤ Rz(2π) ├┤ √X ├»
          ┌─┴─┐   ┌─────────┐     │      └─┬────┬┘ ┌──┴────┴─┐ └─┬───┬──┘└────┘»
  q_1: ───┤ X ├───┤ Rz(π/2) ├─────┼────────┤ √X ├──┤ Rz(π/2) ├───┤ I ├─────────»
          └───┘   └─────────┘   ┌─┴─┐   ┌──┴────┴─┐└──┬────┬─┘┌──┴───┴──┐┌───┐ »
  q_2: ─────────────────────────┤ X ├───┤ Rz(π/2) ├───┤ √X ├──┤ Rz(π/2) ├┤ I ├─»
       ┌─────────┐   ┌────┐  ┌──┴───┴──┐└─────────┘   └────┘  └─────────┘└───┘ »
  q_3: ┤ Rz(π/2) ├───┤ √X ├──┤ Rz(π/2) ├───────────────────────────────────────»
       ├─────────┤   ├────┤  ├─────────┤                                       »
  q_4: ┤ Rz(π/2) ├───┤ √X ├──┤ Rz(π/2) ├───────────────────────────────────────»
       └─────────┘   └────┘  └─────────┘                                       »
syn: 2/════════════════════════════════════════════════════════════════════════»
          

In [9]:
q = QuantumRegister(5)
syn = ClassicalRegister(2)
qc = QuantumCircuit(q, syn)

prepare_state(qc, "0")
qc.x(1)
extract_syndrome(qc, syn)
#decode(qc)

#measure(qc)

TypeError: extract_syndrome() missing 1 required positional argument: 'syn'

In [ ]:
sim = AerSimulator(noise_model=noise_model)

compiled = transpile(qc, sim)#, optimization_level=0)

result = sim.run(compiled, shots=1024).result()

In [ ]:
counts = result.get_counts()
print(counts)

In [ ]:
print(qc.draw())

In [ ]:
import qiskit
import qiskit_aer

print(qiskit.__version__)
print(qiskit_aer.__version__)